# Notebook 2a -- Flow matching (~45 min)

In this track notebook we train a flow-matching (FM) model on the
spiral by regressing a *velocity field* directly, then run a quantitative
step-count study against a diffusion baseline.  Conceptually, this completes
the velocity view of transport and sets up Notebook 2c: straighter paths
need fewer steps.

The diffusion baseline is a pre-trained checkpoint, loaded below;
Notebook 1 is not assumed.

Three markers:
- ✏️ marks an exercise
- 📦 marks provided code or context (just read/run)
- ⭐ marks optional extra material

## 0. Setup (📦)

Three terms we will use throughout:

- **NFE** is the number of neural-network forward evaluations per generated
  sample.  It is the cost axis of everything we measure in this notebook.
- The **score** is $s(x, t) := \nabla_x \log p_t(x)$, the gradient of the
  log-density along the noising path.
- The **PF-ODE** is the probability-flow ODE (ordinary differential
  equation): the deterministic ODE whose flow reproduces the same marginals
  $p_t$ as the diffusion SDE (stochastic differential equation).  It is how
  we sample the diffusion baseline here.

In [ ]:
from functools import partial
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import flax.nnx as nnx
import optax

import matplotlib.pyplot as plt
from tqdm import tqdm

import iaifi_gm as gm  # helper package

rngs = nnx.Rngs(0)
gm.plotting.use_style()
gm.plotting.device_report()

# Repo-relative paths (works from the repo root or from notebooks/).
REPO = Path.cwd() if (Path.cwd() / "checkpoints").exists() else Path.cwd().parent
CKPT_DIR = REPO / "checkpoints"

📦 Our diffusion baseline is the **spiral diffusion teacher** checkpoint,
trained by `scripts/train_spiral_diffusion.py` -- the same file Notebook 2c
uses as its distillation teacher.  It is an $\varepsilon$-prediction network
$\varepsilon_\theta(x_t, t)$ trained on the trig variance-preserving path.

In [ ]:
TEACHER_CKPT = CKPT_DIR / "spiral_diffusion.msgpack"
assert TEACHER_CKPT.exists(), (
    "Teacher checkpoint not found -- run `python "
    "scripts/train_spiral_diffusion.py` from the repo root (~15 s) to create it."
)
teacher = gm.models.TimeMLP(dim=2, hidden=128, depth=3, time_dim=32, rngs=nnx.Rngs(params=0))
teacher = gm.checkpoints.load(teacher, TEACHER_CKPT)
print("loaded diffusion teacher from", TEACHER_CKPT)

## 1. From score to velocity

In Notebook 1, the diffusion model learned $\varepsilon$ (equivalently the score)
along a *fixed Gaussian path*, and only converted the score to a velocity at
sampling time, inside the PF-ODE sampler.  Flow matching eliminates the detour.
Parametrize the **velocity field** $v_\theta(x_t, t)$ and regress it directly.

**Convention:** same clock as Notebook 1 -- $t = 0$ is noise, $t = 1$ is data,
the direction of generation.  (DDPM-style papers -- denoising diffusion
probabilistic models -- run time the other way; their $t$ is our $1 - t$.)

The conditional path per data point is the **linear (rectified-flow)
interpolant**:

$$ x_t = t\,x + (1 - t)\,z, \qquad x \sim p_{\text{data}},\quad z \sim N(0, I), $$

with conditional velocity target

$$ u = \frac{d x_t}{dt} = x - z . $$

Note that this is a linear schedule $(\alpha_t, \sigma_t) = (t, 1 - t)$
that is *not* variance-preserving, $\alpha_t^2 + \sigma_t^2 \neq 1$.

**Why regressing the conditional target works (CFM -- conditional flow
matching).**  The per-sample target $x - z$ is *not* the marginal velocity,
but minimizing the MSE (mean squared error) against it has the same
optimum: the MSE minimizer is the conditional expectation, and
$\mathbb{E}[x - z \mid x_t] = v(x_t, t)$, so the two losses differ by a
θ-independent constant ([Lipman et al., 2022](https://arxiv.org/abs/2210.02747)).
This is the same one-line argument as Notebook 1's denoising-score-matching
identity; see the companion post, *Tutorial: Generative models as transport*.

**A word on straightness**, since it drives §3 and §4: the *conditional*
paths are straight lines by construction, but the *marginal* ODE
trajectories -- what the sampler actually integrates -- are only straighter
than the diffusion parametrization's, not straight.  For a multimodal target
they are necessarily curved.

📦 The cell below draws the two conditional paths for the same six $(z, x)$
pairs: linear (FM) on the left, trig-VP (diffusion) on the right.  Both
share the endpoints, only the schedules differ.

In [ ]:
key_xd, key_zd = jax.random.split(rngs(), 2)
x_demo = gm.targets.sample_spiral(key_xd, 6)
z_demo = jax.random.normal(key_zd, (6, 2))

t_path = jnp.linspace(0.0, 1.0, 100)[:, None, None]                 # (T, 1, 1)
path_lin = t_path * x_demo + (1 - t_path) * z_demo                  # (T, 6, 2)
path_vp = (
    jnp.sin(jnp.pi * t_path / 2) * x_demo + jnp.cos(jnp.pi * t_path / 2) * z_demo
)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, path, title in [
    (axes[0], path_lin, r"linear (FM): $x_t = t\,x + (1-t)\,z$"),
    (axes[1], path_vp, r"trig VP (diffusion): $x_t = \alpha_t x + \sigma_t z$"),
]:
    path = np.asarray(path)
    ax.plot(path[..., 0], path[..., 1], lw=1.0, alpha=0.8)
    ax.scatter(*np.asarray(z_demo).T, marker="o", facecolors="none", edgecolors="k", label="$z$ (noise)")
    ax.scatter(*np.asarray(x_demo).T, marker="*", s=80, c="k", label="$x$ (data)")
    ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
    ax.set_title(title)
axes[0].legend(loc="upper left")
fig.tight_layout()

## 2. Implement & train

📦 We reuse the model class from Notebook 1 (`gm.models.TimeMLP`, an MLP
with a sinusoidal time embedding, called as `model(x, t)`) and the same
train-step scaffold.  Only the loss changes.

In [ ]:
model = gm.models.TimeMLP(dim=2, hidden=128, depth=3, time_dim=32, rngs=nnx.Rngs(params=1))

### ✏️ Exercise 1 -- the CFM loss

Implement exactly

$$ L(\theta) = \mathbb{E}_{t \sim U(0,1),\; x \sim p_{\text{data}},\; z \sim N(0,I)}
   \big\lVert v_\theta(x_t, t) - (x - z) \big\rVert^2 ,
   \qquad x_t = t\,x + (1 - t)\,z . $$

Sample `t ~ U(0, 1)` with `key_t` (shape `(B,)`), sample `z` with `key_z`
(shape of `x`), form `x_t`, and return the mean squared error of
`model(x_t, t)` against `x - z`.

The shapes are `x: (B, 2)`, `t: (B,)`, `z: (B, 2)`, and the loss is a
scalar.

In [ ]:
def cfm_loss(model, key_t, key_z, x):
    """Conditional flow-matching loss.

    x: (B, 2); t: (B,) ~ U(0,1) from key_t; z: (B, 2) ~ N(0,I) from key_z
    -> scalar loss.
    """
    raise NotImplementedError  # YOUR CODE HERE

In [ ]:
# 📦 Shape check -- run before training.
x_test = gm.targets.sample_spiral(jax.random.key(0), 512)
v_test = model(x_test, jnp.full(512, 0.5))
assert v_test.shape == (512, 2), f"model output shape {v_test.shape}, expected (512, 2)"
loss_test = cfm_loss(model, jax.random.key(1), jax.random.key(2), x_test)
assert jnp.shape(loss_test) == (), f"loss must be a scalar, got shape {jnp.shape(loss_test)}"
assert bool(jnp.isfinite(loss_test)), "loss is not finite"
assert 1.5 < float(loss_test) < 4.0, (
    f"untrained loss should be O(1) (E[|x - z|^2] per dim), got {float(loss_test):.2f}"
)
print(f"untrained CFM loss: {float(loss_test):.2f}")

📦 Training uses the same scaffold as Notebook 1, for ~4000 steps -- a few
seconds (tens of seconds on a slower laptop).

In [ ]:
N_STEPS = 4000
BATCH = 256

optimizer = nnx.Optimizer(
    model, optax.adam(optax.cosine_decay_schedule(1e-3, N_STEPS)), wrt=nnx.Param
)


@nnx.jit
def train_step(model, optimizer, key):
    key_x, key_t, key_z = jax.random.split(key, 3)
    x = gm.targets.sample_spiral(key_x, BATCH)
    loss, grads = nnx.value_and_grad(cfm_loss)(model, key_t, key_z, x)
    optimizer.update(grads=grads, model=model)
    return loss


losses = np.full(N_STEPS, np.nan)
keys = jax.random.split(rngs(), N_STEPS)
for i in tqdm(range(N_STEPS), desc="CFM"):
    losses[i] = train_step(model, optimizer, keys[i])
print(f"final loss (mean of last 100): {losses[-100:].mean():.3f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(losses)
ax.set_xlabel("step")
ax.set_ylabel("CFM loss")
fig.tight_layout()

Just as for the diffusion case, the loss plateaus at ~1.6 and *cannot* go to zero:
at a given $(x_t, t)$ many $(x, z)$ pairs coexist, and $v_\theta$ can only fit
their conditional mean.

## 3. Sampling & step-count study

Both models define an ODE $\dot x = v(x, t)$: the FM model is its velocity
field, while the diffusion teacher's PF-ODE velocity comes from the
score-to-velocity conversion, as in Notebook 1's sampler.  For the trig-VP
path, in terms of the ε-network,

$$ v(x, t) = a_t\, x + b_t\, \varepsilon_\theta(x, t), \qquad
   a_t = \tfrac{\pi}{2} \cot\!\big(\tfrac{\pi t}{2}\big), \quad
   b_t = -\tfrac{\pi}{2} \big/ \sin\!\big(\tfrac{\pi t}{2}\big) . $$

📦 The cell below wraps both velocities with a common signature
`(x, t) -> dx/dt`, so one integrator serves both.  The diffusion
coefficients blow up like $1/t$ at the noise end, so its time grid must
start at `T_MIN = 1e-2` (the same clip as its training) while the FM grid
runs the full $[0, 1]$.

In [ ]:
T_MIN = 1e-2  # diffusion grid start/end clip; FM needs no clip


@nnx.jit
def fm_velocity(model, x, t):
    """v(x, t) of the FM model; x: (n, d...), t scalar -> (n, d...)."""
    return model(x, jnp.full(x.shape[0], t))


@nnx.jit
def diffusion_velocity(model, x, t):
    """PF-ODE velocity of the trig-VP diffusion model; x: (n, 2), t scalar -> (n, 2).

    Only valid for t in [T_MIN, 1 - T_MIN] (1/t blow-up at the noise end).
    """
    a = (jnp.pi / 2) / jnp.tan(jnp.pi * t / 2)
    b = -(jnp.pi / 2) / jnp.sin(jnp.pi * t / 2)
    return a * x + b * model(x, jnp.full(x.shape[0], t))


fm_v = partial(fm_velocity, model)
diff_v = partial(diffusion_velocity, teacher)

### ✏️ Exercise 2 -- fixed-step Euler sampler

Implement the Euler update, exactly

$$ x \leftarrow x + (t_{k+1} - t_k)\, v(x, t_k), $$

starting from $x = z \sim N(0, I)$ and marching over the provided time grid
`ts` $= (t_0 < t_1 < \dots < t_K)$ from noise to data.  The NFE of one call
is exactly `len(ts) - 1`.

The state `x` keeps its shape (e.g. `(n, 2)`) throughout, and `v_fn(x, t)`
takes the current state and the *scalar* current time.

In [ ]:
def euler_sample(v_fn, key, ts, shape):
    """Fixed-step Euler integration of dx/dt = v_fn(x, t).

    v_fn: (x: (n, d...), t: scalar) -> (n, d...)
    ts: (K+1,) increasing time grid (noise end first); shape: e.g. (n, 2).
    -> (n, d...) samples at t = ts[-1].
    """
    x = jax.random.normal(key, shape)  # x = z at t = ts[0]
    raise NotImplementedError  # YOUR CODE HERE
    return x

In [ ]:
# 📦 Shape + correctness check -- run before the sweep.
x_out = euler_sample(fm_v, jax.random.key(0), jnp.linspace(0.0, 1.0, 5), (8, 2))
assert x_out.shape == (8, 2), f"output shape {x_out.shape}, expected (8, 2)"
assert bool(jnp.all(jnp.isfinite(x_out))), "samples are not finite"
# One Euler step over [0, 1] must be exactly z + v(z, 0):
z0 = jax.random.normal(jax.random.key(0), (8, 2))
x_one = euler_sample(fm_v, jax.random.key(0), jnp.array([0.0, 1.0]), (8, 2))
assert jnp.allclose(x_one, z0 + fm_v(z0, jnp.array(0.0)), atol=1e-5), (
    "one Euler step != z + v(z, 0) -- check dt and where you evaluate v"
)
print("euler_sample looks right")

📦 Now the step-count sweep: we run both models through your integrator at
NFE ∈ {1, 2, 4, 8, 16, 32}, from the same starting noise.  Quality is
measured by the **energy distance** (`gm.metrics.energy_distance`), a
distance between two sample sets that vanishes exactly when the underlying
distributions coincide; it is deterministic given the samples and has no
tuning knobs.  We print the target-vs-target noise floor alongside: anything
at the floor is statistically indistinguishable from the target.

In [ ]:
STEP_COUNTS = (1, 2, 4, 8, 16, 32)
N_EVAL = 2048

key_eval, key_ref, key_ref2 = jax.random.split(rngs(), 3)
x_target = gm.targets.sample_spiral(key_ref, N_EVAL)
ed_floor = float(gm.metrics.energy_distance(gm.targets.sample_spiral(key_ref2, N_EVAL), x_target))
print(f"energy-distance noise floor (target vs target, n={N_EVAL}): {ed_floor:.4f}")

eds = {"FM": [], "diffusion": []}
n_col = len(STEP_COUNTS) + 1
fig, axes = plt.subplots(2, n_col, figsize=(2.0 * n_col, 5.0))
for i in range(2):  # leftmost column: the target, for visual reference
    gm.plotting.scatter2d(x_target, ax=axes[i, 0], s=1.5, alpha=0.3, color="grey")
    axes[i, 0].set_title("target", fontsize=8)
    axes[i, 0].set_xticks([]); axes[i, 0].set_yticks([])
for j, steps in enumerate(STEP_COUNTS, start=1):
    grids = {
        "FM": jnp.linspace(0.0, 1.0, steps + 1),
        "diffusion": jnp.linspace(T_MIN, 1 - T_MIN, steps + 1),
    }
    for i, (name, v_fn) in enumerate([("FM", fm_v), ("diffusion", diff_v)]):
        x_s = euler_sample(v_fn, key_eval, grids[name], (N_EVAL, 2))
        ed = float(gm.metrics.energy_distance(x_s, x_target))
        eds[name].append(ed)
        gm.plotting.scatter2d(x_s, ax=axes[i, j], s=1.5, alpha=0.3)
        axes[i, j].set_title(f"{name}, NFE {steps}\nED {ed:.3f}", fontsize=8)
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
fig.tight_layout(h_pad=2.0)

print(f"{'NFE':>5} {'FM':>10} {'diffusion':>10}")
for steps, e_fm, e_di in zip(STEP_COUNTS, eds["FM"], eds["diffusion"]):
    print(f"{steps:>5} {e_fm:>10.4f} {e_di:>10.4f}")

assert eds["FM"][-1] < 0.05, (
    "FM samples at 32 steps are far from the target -- most likely the CFM "
    "loss in Exercise 1 (check the t vs 1-t weights in x_t and the sign of "
    "the target x - z)"
)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(STEP_COUNTS, eds["FM"], "o-", label="FM")
ax.plot(STEP_COUNTS, eds["diffusion"], "s-", label="diffusion (PF-ODE)")
ax.axhline(ed_floor, color="k", ls="--", lw=1, label="sampling noise floor")
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xticks(STEP_COUNTS, [str(s) for s in STEP_COUNTS])
ax.set_xlabel("NFE (Euler steps)")
ax.set_ylabel("energy distance to target")
ax.legend()
fig.tight_layout()

📦 To see *why* few-step FM wins, watch the marginal trajectories -- the
thing the sampler actually integrates.  Both models start from the same noise; the
grey dots are target samples.

In [ ]:
N_TRAJ, TRAJ_STEPS = 12, 64


def euler_trajectory(v_fn, x0, ts):
    """Euler integration recording every state; x0: (n, 2) -> (K+1, n, 2)."""
    xs, x = [x0], x0
    for t, t_next in zip(ts[:-1], ts[1:]):
        x = x + (t_next - t) * v_fn(x, t)
        xs.append(x)
    return np.asarray(jnp.stack(xs))


z_traj = jax.random.normal(jax.random.key(3), (N_TRAJ, 2))
traj = {
    "FM": euler_trajectory(fm_v, z_traj, jnp.linspace(0.0, 1.0, TRAJ_STEPS + 1)),
    "diffusion": euler_trajectory(diff_v, z_traj, jnp.linspace(T_MIN, 1 - T_MIN, TRAJ_STEPS + 1)),
}

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, name in zip(axes, ["FM", "diffusion"]):
    gm.plotting.scatter2d(x_target[:1024], ax=ax, s=1.5, alpha=0.15, color="grey")
    ax.plot(traj[name][..., 0], traj[name][..., 1], lw=1.6, alpha=0.9, zorder=2)
    ax.scatter(*z_traj.T, marker="o", facecolors="none", edgecolors="k", s=25, zorder=3)
    ax.scatter(*traj[name][-1].T, marker="*", c="k", s=60, zorder=3)
    ax.set_title(f"{name}: marginal ODE trajectories")
fig.tight_layout()

The FM trajectories are visibly straighter than the diffusion ones, but they
are *not* straight: the marginal flow of a multimodal target must bend, even
though every conditional path was a straight line.

The low-NFE gap mixes two effects: FM's straighter marginal paths,
and the stiffness of the diffusion ε/score parametrization near the noise end,
where $\alpha \to 0$ and the $1/t$-style coefficients punish coarse Euler grids
(the reason for the `T_MIN` clip).
The plot shows the combined improvement, not curvature alone.
It is also specifically a *low-NFE* win: by ~32 steps both are near the
floor, and the teacher (trained 3× longer than the FM run here) may
even win there.

## 4. Discussion & pointers

**One family, two members.**  Both models were trained by the same
fixed-path, simulation-free recipe -- corrupt data along a prescribed
$(\alpha_t, \sigma_t)$ path, regress a network against a closed-form
per-sample target -- and differ only in schedule and regression target:

|  | path $(\alpha_t, \sigma_t)$ | network regresses | sampling | typical NFE (here) |
|---|---|---|---|---|
| diffusion (NB1) | trig VP: $(\sin\frac{\pi t}{2}, \cos\frac{\pi t}{2})$ | $\varepsilon_\theta \approx z$ | PF-ODE via the score (or ancestral SDE) | ~50-128 |
| flow matching | linear: $(t, 1-t)$ | $v_\theta \approx x - z$ | ODE directly | ~8-32 |

Conditional paths are straight by construction;
the *marginal* ODE is only straighter.  **Reflow** (rectified flow) closes the gap:
 generate $(z, x)$ pairs *from the trained model itself*
-- $z$ and where the ODE transports it -- and retrain on this coupling.
Re-coupled pairs cross far less, and iterating reflow drives the marginal
flow toward straight one-step transport
([Liu et al., 2022](https://arxiv.org/abs/2209.03003)).

State-of-the-art image and video generators train
exactly this objective at scale; for the full design space (paths,
couplings, guidance) see the Flow Matching Guide and Code
([Lipman et al., 2024](https://arxiv.org/abs/2412.06264)).

Notebook 2c's distillation learns the *integrated map* $z \mapsto x$
instead of its velocity field.

## ⭐ Stretch 1 -- fashion-MNIST flow matching

The same loss and the same integrator work for images.  We provide a
UNet checkpoint trained with the CFM loss on fashion-MNIST
(`scripts/train_fmnist_fm.py`; the state $x$ has shape `(B, 28, 28, 1)` with
values in $[-1, 1]$).  Rerun the step-count sweep and watch the sample grids
sharpen with NFE.  The cell skips if the checkpoint is not on disk.

In [ ]:
FMNIST_CKPT = CKPT_DIR / "fmnist_fm.msgpack"
if not FMNIST_CKPT.exists():
    print(f"[skip] {FMNIST_CKPT} not found -- fashion-MNIST stretch needs the "
          "pretrained checkpoint (see scripts/train_fmnist_fm.py). Nothing below depends on it.")
else:
    unet = gm.models.SmallUNet(channels=(32, 64, 128), time_dim=128, rngs=nnx.Rngs(params=0))
    unet = gm.checkpoints.load(unet, FMNIST_CKPT)
    unet_v = partial(fm_velocity, unet)  # same wrapper: model(x, t) with batched t

    for steps in (1, 4, 32):
        imgs = euler_sample(
            unet_v, jax.random.key(0), jnp.linspace(0.0, 1.0, steps + 1), (16, 28, 28, 1)
        )
        fig = gm.plotting.image_grid(imgs, nrow=8)
        fig.suptitle(f"fashion-MNIST FM, NFE {steps}", y=1.02)

    # Optional pixel-space energy distance vs real images
    # (first call downloads the fashion-MNIST test split, ~4.5 MB).
    images, _ = gm.data.load_fashion_mnist(split="test", data_dir=REPO / "data")
    x_real = images[:256].reshape(256, -1)
    x_gen = np.asarray(
        euler_sample(unet_v, jax.random.key(1), jnp.linspace(0.0, 1.0, 33), (256, 28, 28, 1))
    ).reshape(256, -1)
    print(f"pixel-space energy distance @ NFE 32: {float(gm.metrics.energy_distance(x_gen, x_real)):.4f}")
    print(f"real-vs-real floor (n=256):          "
          f"{float(gm.metrics.energy_distance(x_real, images[256:512].reshape(256, -1))):.4f}")

## ⭐ Stretch 2 -- mini-batch OT coupling (teaser)

CFM pairs each data point with an *independent* noise draw, so conditional
paths cross each other -- and that crossing is exactly what forces the
marginal flow to bend.  **OT** (optimal transport) coupling reduces it:
within each batch, re-pair $z \leftrightarrow x$ by the assignment
minimizing total squared distance *before* interpolating.  Straighter,
shorter conditional paths mean less averaging, and hence a straighter
marginal flow ([Tong et al., 2023](https://arxiv.org/abs/2302.00482)).
The 📦 cell below visualizes the pairings.  To train with this coupling,
draw $z$ first, re-pair $x \leftarrow x[\mathrm{col}]$ as below, and write
a variant of `cfm_loss` that accepts the paired $(z, x)$ batch -- try it.

In [ ]:
try:
    from scipy.optimize import linear_sum_assignment
except ImportError:
    print("[skip] scipy not available -- OT-coupling teaser needs scipy.optimize.")
else:
    key_xo, key_zo = jax.random.split(jax.random.key(7), 2)
    x_b = gm.targets.sample_spiral(key_xo, 64)
    z_b = jax.random.normal(key_zo, (64, 2))
    cost = jnp.sum((z_b[:, None, :] - x_b[None, :, :]) ** 2, axis=-1)  # (64, 64)
    _, col = linear_sum_assignment(np.asarray(cost))                   # z_i -> x_{col[i]}

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    for ax, x_paired, title in [
        (axes[0], x_b, "independent coupling"),
        (axes[1], x_b[col], "mini-batch OT coupling"),
    ]:
        seg = np.stack([np.asarray(z_b), np.asarray(x_paired)])  # (2, 64, 2)
        ax.plot(seg[..., 0], seg[..., 1], lw=0.8, alpha=0.6, color="C0")
        ax.scatter(*np.asarray(z_b).T, marker="o", facecolors="none", edgecolors="k", s=15)
        ax.scatter(*np.asarray(x_paired).T, marker="*", c="k", s=30)
        cost_pair = float(jnp.mean(jnp.sum((x_paired - z_b) ** 2, axis=-1)))
        ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
        ax.set_title(f"{title}\nmean squared pair length {cost_pair:.2f}")
    fig.tight_layout()